# EarningsLens — Step 1: Data Collection

## What are we doing in this step?

Our transformer needs TWO things to learn from:
1. **Earnings call transcripts** — what the CEO actually said
2. **Stock price labels** — did the stock go UP, DOWN, or stay FLAT 48 hours after the call?

Think of it like this:
- Input to model → CEO said: *'we remain cautiously optimistic'*
- Label → stock dropped 3% two days later → label = **DOWN**

After collecting thousands of these pairs, our transformer learns the pattern.

## Plan for this notebook:
1. Install all required libraries
2. Download earnings call transcripts from SEC EDGAR
3. Pull matching stock prices using yfinance
4. Create labels (UP / DOWN / FLAT)
5. Save everything to a clean CSV file ready for Step 2

---

## Cell 1 — Install Libraries

We need these libraries:
- `sec-edgar-downloader` → downloads filings from SEC EDGAR website
- `yfinance` → pulls stock price data from Yahoo Finance
- `pandas` → organizes data into tables
- `beautifulsoup4` → parses HTML from SEC filings to extract clean text
- `requests` → makes HTTP requests to SEC API
- `tqdm` → shows progress bars so you know how much is done

In [ ]:
# Install all required libraries
# Run this cell only once — you don't need to re-run every time

!pip install sec-edgar-downloader yfinance pandas beautifulsoup4 requests tqdm lxml

## Cell 2 — Import Everything

Now we import all the libraries we just installed.
We also set up a folder structure to save our data.

In [ ]:
# ── Standard library imports ──────────────────────────────────────────────────
import os                        # for creating folders and file paths
import re                        # for cleaning text using regex patterns
import time                      # for adding delays between API requests
import json                      # for reading JSON responses from APIs
from datetime import datetime, timedelta
from pathlib import Path         # cleaner way to handle file paths

# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd              # tables and data manipulation
import numpy as np               # numerical operations

# ── Web scraping ──────────────────────────────────────────────────────────────
import requests                  # making HTTP requests to APIs
from bs4 import BeautifulSoup    # parsing HTML content from SEC filings
from tqdm import tqdm            # progress bars

# ── Stock price data ──────────────────────────────────────────────────────────
import yfinance as yf            # Yahoo Finance stock price API

# ── Warnings ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Cell 3 — Set Up Folder Structure

We create folders to save:
- Raw transcripts (text files from SEC)
- Processed data (cleaned + labeled CSV)

Keeping data organized now saves hours of confusion later.

In [ ]:
# ── Create project folder structure ───────────────────────────────────────────

# Base project directory — change this to wherever you want to save data
BASE_DIR = Path("earningslens_data")

# Sub-folders
RAW_DIR       = BASE_DIR / "raw_transcripts"    # raw text files from SEC
PROCESSED_DIR = BASE_DIR / "processed"          # cleaned CSV files
LOGS_DIR      = BASE_DIR / "logs"               # error logs

# Create all folders (exist_ok=True means no error if they already exist)
for folder in [BASE_DIR, RAW_DIR, PROCESSED_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Folder structure created:")
print(f"   {BASE_DIR}/")
print(f"   ├── raw_transcripts/   ← SEC filing text files go here")
print(f"   ├── processed/         ← cleaned CSV datasets go here")
print(f"   └── logs/              ← error logs go here")

✅ Folder structure created:
   earningslens_data/
   ├── raw_transcripts/   ← SEC filing text files go here
   ├── processed/         ← cleaned CSV datasets go here
   └── logs/              ← error logs go here


## Cell 4 — Define Our Target Companies

We pick a list of S&P 500 companies to collect transcripts for.

**Why these companies?**
- They are large, liquid stocks so price movement is meaningful
- They hold regular quarterly earnings calls — 4 per year each
- 20 companies × 4 calls/year × 6 years = ~480 transcripts to start

You can expand this list later — we start small to test everything works.

In [ ]:
# ── Target companies ───────────────────────────────────────────────────────────
# Format: {"ticker": "AAPL", "name": "Apple Inc", "sector": "Technology"}
# We cover multiple sectors so our model learns across different industries

COMPANIES = [
    # Technology
    {"ticker": "AAPL",  "name": "Apple Inc",            "sector": "Technology"},
    {"ticker": "MSFT",  "name": "Microsoft Corp",        "sector": "Technology"},
    {"ticker": "GOOGL", "name": "Alphabet Inc",          "sector": "Technology"},
    {"ticker": "META",  "name": "Meta Platforms",        "sector": "Technology"},
    {"ticker": "NVDA",  "name": "NVIDIA Corp",           "sector": "Technology"},

    # Finance
    {"ticker": "JPM",   "name": "JPMorgan Chase",        "sector": "Finance"},
    {"ticker": "BAC",   "name": "Bank of America",       "sector": "Finance"},
    {"ticker": "GS",    "name": "Goldman Sachs",         "sector": "Finance"},

    # Healthcare
    {"ticker": "JNJ",   "name": "Johnson & Johnson",     "sector": "Healthcare"},
    {"ticker": "PFE",   "name": "Pfizer Inc",            "sector": "Healthcare"},
    {"ticker": "UNH",   "name": "UnitedHealth Group",    "sector": "Healthcare"},

    # Consumer
    {"ticker": "AMZN",  "name": "Amazon.com Inc",        "sector": "Consumer"},
    {"ticker": "WMT",   "name": "Walmart Inc",           "sector": "Consumer"},
    {"ticker": "MCD",   "name": "McDonald's Corp",       "sector": "Consumer"},

    # Energy
    {"ticker": "XOM",   "name": "Exxon Mobil",           "sector": "Energy"},
    {"ticker": "CVX",   "name": "Chevron Corp",          "sector": "Energy"},

    # Industrial
    {"ticker": "BA",    "name": "Boeing Co",             "sector": "Industrial"},
    {"ticker": "CAT",   "name": "Caterpillar Inc",       "sector": "Industrial"},

    # Telecom
    {"ticker": "T",     "name": "AT&T Inc",              "sector": "Telecom"},
    {"ticker": "VZ",    "name": "Verizon Communications","sector": "Telecom"},
]

# Date range for transcript collection
# 6 years × 4 quarters × 20 companies = ~480 transcripts
START_YEAR = 2018
END_YEAR   = 2024

print(f"✅ Defined {len(COMPANIES)} companies across {len(set(c['sector'] for c in COMPANIES))} sectors")
print(f"   Date range: {START_YEAR} → {END_YEAR}")
print(f"   Estimated transcripts: ~{len(COMPANIES) * 4 * (END_YEAR - START_YEAR)}")

✅ Defined 20 companies across 7 sectors
   Date range: 2018 → 2024
   Estimated transcripts: ~480


## Cell 5 — SEC EDGAR API: Find Earnings Call Filings

**How SEC EDGAR works:**
- Companies must file official documents with the SEC (US financial regulator)
- Earnings call transcripts are filed as **8-K forms** (current report)
- SEC has a free public API we can use — no login needed

We will:
1. Search for each company's 8-K filings
2. Filter ones that contain earnings call transcripts
3. Download the text content

### Provide your Name and Email for SEC EDGAR API

To comply with SEC EDGAR API usage policy, you must provide a valid `User-Agent` string including your name and email. Please enter your details below. This information will only be used for API requests and will not be stored.

In [ ]:
# ── Dynamically update SEC_HEADERS ────────────────────────────────────────────

# @title Enter your Name and Email
user_name = "hamza" # @param {type:"string"}
user_email = "hamzshahid8587@gmail.com" # @param {type:"string"}

if user_name and user_email:
    SEC_HEADERS["User-Agent"] = f"{user_name} {user_email}"
    print(f"✅ User-Agent updated to: {SEC_HEADERS['User-Agent']}")
else:
    print("⚠️ Please enter your name and email to proceed with SEC EDGAR requests.")



✅ User-Agent updated to: hamza hamzshahid8587@gmail.com


In [ ]:
# ── SEC EDGAR API Helper Functions ────────────────────────────────────────────

# SEC requires a User-Agent header with your name and email
# This is their policy — always include it or requests get blocked
SEC_HEADERS = {
    "User-Agent": "EarningsLens Research yourname@email.com",  # ← CHANGE THIS TO YOUR NAME AND EMAIL
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}


def get_company_cik(ticker: str) -> str:
    """
    Every company on SEC EDGAR has a unique ID called CIK (Central Index Key).
    This function converts a stock ticker (e.g. 'AAPL') to its CIK number.
    We need the CIK to search for that company's filings.
    """
    # A more robust way to get CIK: Use SEC's company search to find the CIK
    # This often involves an indirect lookup or a dedicated endpoint.
    # The previous `company_tickers.json` and `company_tickers_exchange.json` are unreliable.
    # We'll use the 'ticker-to-CIK' lookup endpoint if available, or fall back to searching.

    # The SEC provides a mapping from ticker to CIK via the `CIK_lookup_from_ticker` endpoint.
    # This is a more stable way to get the CIK.
    lookup_url = f"https://api.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={ticker}&type=&dateb=&owner=only&start=0&count=1000&output=xml"

    try:
        response = requests.get(lookup_url, headers=SEC_HEADERS, timeout=10)

        # --- DEBUGGING PRINTS --- #
        print(f"   DEBUG: SEC API Status Code for {ticker}: {response.status_code}")
        print(f"   DEBUG: SEC API Response Text for {ticker} (first 500 chars): {response.text[:500]}")
        # --- END DEBUGGING PRINTS --- #

        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

        # Parse the XML response to find the CIK
        soup = BeautifulSoup(response.content, "lxml")
        cik_tag = soup.find('cik') # CIK is usually within a <cik> tag in XML

        if cik_tag and cik_tag.string:
            cik = str(cik_tag.string).zfill(10)
            return cik

        print(f"   ⚠️  Ticker {ticker} not found in SEC database via new lookup.")
        return None

    except requests.exceptions.HTTPError as errh:
        print(f"   ❌ HTTP Error for {ticker}: {errh}")
        return None
    except requests.exceptions.ConnectionError as errc:
        print(f"   ❌ Error Connecting for {ticker}: {errc}")
        return None
    except requests.exceptions.Timeout as errt:
        print(f"   ❌ Timeout Error for {ticker}: {errt}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"   ❌ General Request Error for {ticker}: {err}")
        return None
    except Exception as e:
        print(f"   ❌ An unexpected error occurred for {ticker}: {e}")
        return None


def get_8k_filings(cik: str, ticker: str, start_year: int, end_year: int) -> list:
    """
    Fetches all 8-K filings for a company between start_year and end_year.
    8-K is the form companies file for major events — including earnings calls.
    Returns a list of filing metadata dictionaries.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    try:
        response = requests.get(url, headers=SEC_HEADERS, timeout=15)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()

        filings = []

        # SEC returns filings in two groups: recent and older
        # We check both to get full date range coverage
        filing_data = data.get("filings", {}).get("recent", {})

        forms       = filing_data.get("form", [])
        dates       = filing_data.get("filingDate", [])
        accessions  = filing_data.get("accessionNumber", [])
        descriptions= filing_data.get("primaryDocument", [])

        for i, form in enumerate(forms):
            # Only keep 8-K filings (earnings call reports)
            if form != "8-K":
                continue

            # Only keep filings in our date range
            filing_year = int(dates[i][:4])
            if not (start_year <= filing_year <= end_year):
                continue

            filings.append({
                "ticker":    ticker,
                "cik":       cik,
                "date":      dates[i],
                "accession": accessions[i].replace("-", ""),  # clean format
                "document":  descriptions[i]
            })

        return filings

    except requests.exceptions.HTTPError as errh:
        print(f"   ❌ HTTP Error for {ticker}: {errh}")
        return []
    except requests.exceptions.ConnectionError as errc:
        print(f"   ❌ Error Connecting for {ticker}: {errc}")
        return []
    except requests.exceptions.Timeout as errt:
        print(f"   ❌ Timeout Error for {ticker}: {errt}")
        return []
    except requests.exceptions.RequestException as err:
        print(f"   ❌ General Request Error for {ticker}: {err}")
        return []
    except json.JSONDecodeError as e:
        print(f"   ❌ JSON Decoding Error for {ticker}: {e}")
        return []
    except Exception as e:
        print(f"   ❌ An unexpected error occurred for {ticker}: {e}")
        return []


print("✅ SEC EDGAR helper functions defined")

# Quick test — look up Apple's CIK
print("\n🔍 Testing with Apple (AAPL)...")
test_cik = get_company_cik("AAPL")
print(f"   Apple CIK: {test_cik}")

✅ SEC EDGAR helper functions defined

🔍 Testing with Apple (AAPL)...
   ❌ Error Connecting for AAPL: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb09db20>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))
   Apple CIK: None


## Cell 6 — Extract Transcript Text from Filings

Once we have the filing metadata, we need to:
1. Download the actual HTML document from SEC
2. Use BeautifulSoup to extract clean readable text
3. Detect if this 8-K is actually an earnings call (not just any company announcement)

**How do we know if a filing is an earnings call?**
We look for keywords in the text like 'earnings call', 'quarterly results', 'conference call'

In [ ]:
# ── Transcript Extraction Functions ───────────────────────────────────────────

# Keywords that indicate this 8-K filing contains an earnings call transcript
# If none of these appear in the document, it's a different kind of announcement
EARNINGS_KEYWORDS = [
    "earnings call", "quarterly results", "conference call",
    "fourth quarter", "third quarter", "second quarter", "first quarter",
    "q4 ", "q3 ", "q2 ", "q1 ",
    "fiscal year", "full year results"
]


def download_filing_text(cik: str, accession: str, document: str) -> str:
    """
    Downloads the actual text content of an SEC filing.

    SEC stores filings at a URL like:
    https://www.sec.gov/Archives/edgar/data/{CIK}/{ACCESSION}/{DOCUMENT}

    We download it, parse the HTML, and return clean text.
    """
    # Remove leading zeros from CIK for the URL
    cik_clean = str(int(cik))

    # Format accession number with dashes for the URL path
    accession_dashed = f"{accession[:10]}-{accession[10:12]}-{accession[12:]}"

    url = f"https://www.sec.gov/Archives/edgar/data/{cik_clean}/{accession}/{document}"

    try:
        # Add a small delay to be respectful of SEC's servers
        time.sleep(0.1)

        response = requests.get(url, headers=SEC_HEADERS, timeout=20)

        if response.status_code != 200:
            return None

        # Parse HTML and extract all text
        soup = BeautifulSoup(response.content, "lxml")

        # Remove script and style tags — we don't want code in our text
        for tag in soup(["script", "style", "table"]):
            tag.decompose()

        # Get plain text
        text = soup.get_text(separator=" ", strip=True)

        # Basic cleaning
        text = re.sub(r'\s+', ' ', text)           # collapse multiple spaces
        text = re.sub(r'\n+', '\n', text)          # collapse multiple newlines
        text = text.strip()

        return text

    except Exception as e:
        return None


def is_earnings_call(text: str) -> bool:
    """
    Checks if a filing text is actually an earnings call transcript.
    Returns True if earnings-related keywords are found.

    Why do we need this?
    Companies file 8-K forms for many events (mergers, CEO changes, etc.)
    We only want the quarterly earnings call ones.
    """
    text_lower = text.lower()

    # Count how many earnings keywords appear
    keyword_hits = sum(1 for kw in EARNINGS_KEYWORDS if kw in text_lower)

    # Require at least 2 keywords to be confident it's an earnings call
    return keyword_hits >= 2


def extract_ceo_remarks(text: str) -> str:
    """
    Earnings calls have two sections:
    1. Prepared remarks (CEO/CFO read from a script) — MOST SIGNAL-RICH
    2. Q&A session (analysts ask questions)

    We try to extract just the prepared remarks section.
    If we can't find the boundary, we return the full text.
    """
    text_lower = text.lower()

    # Common phrases that mark the start of Q&A section
    qa_markers = [
        "question-and-answer", "q&a session", "questions and answers",
        "open the floor", "we'll now take questions", "operator: thank you",
        "[operator instructions]"
    ]

    # Find where Q&A starts
    qa_start = len(text)  # default: use full text

    for marker in qa_markers:
        idx = text_lower.find(marker)
        if idx != -1 and idx < qa_start:
            qa_start = idx

    # Return just the prepared remarks (before Q&A)
    prepared_remarks = text[:qa_start].strip()

    # If extracted section is too short, return full text
    if len(prepared_remarks) < 500:
        return text

    return prepared_remarks


print("✅ Transcript extraction functions defined")

✅ Transcript extraction functions defined


## Cell 7 — Stock Price Label Generator

Now we build the function that creates our training labels.

**Logic:**
- Get closing price on the day of the earnings call
- Get closing price 2 trading days later
- Calculate % change
- If > +1.5% → label = **UP**
- If < -1.5% → label = **DOWN**  
- Otherwise → label = **FLAT**

Why 1.5% threshold? Smaller moves could just be normal market noise, not a reaction to the earnings call specifically.

In [ ]:
# ── Stock Price Label Generator ───────────────────────────────────────────────

# How much price must move to count as UP or DOWN
# Moves smaller than this are labeled FLAT (could be random noise)
PRICE_THRESHOLD = 0.015   # 1.5%

# How many trading days after the call to measure price change
DAYS_AFTER = 2            # ~48 hours (2 trading days)


def get_price_label(ticker: str, filing_date: str) -> dict:
    """
    Given a stock ticker and the date of an earnings call,
    calculates what the stock did 2 trading days later.

    Returns a dict with:
    - price_before: closing price on filing date
    - price_after:  closing price 2 days later
    - pct_change:   percentage price change
    - label:        'UP', 'DOWN', or 'FLAT'
    """
    try:
        # Parse the filing date
        call_date = datetime.strptime(filing_date, "%Y-%m-%d")

        # We need a window of ~10 days to ensure we capture 2 trading days
        # (weekends and holidays mean we can't just add 2 calendar days)
        window_start = call_date - timedelta(days=3)   # 3 days before (buffer)
        window_end   = call_date + timedelta(days=10)  # 10 days after (buffer)

        # Download price history for this window
        stock = yf.Ticker(ticker)
        hist = stock.history(
            start=window_start.strftime("%Y-%m-%d"),
            end=window_end.strftime("%Y-%m-%d")
        )

        # Check we got data
        if hist.empty or len(hist) < DAYS_AFTER + 1:
            return None

        # Reset index so date is a column we can filter
        hist.index = pd.to_datetime(hist.index).tz_localize(None)

        # Find the closest trading day ON or AFTER the filing date
        call_date_naive = pd.Timestamp(call_date)
        future_dates = hist.index[hist.index >= call_date_naive]

        if len(future_dates) < DAYS_AFTER + 1:
            return None

        # Price on the day of the call (or next trading day)
        price_before = hist.loc[future_dates[0], "Close"]

        # Price 2 trading days later
        price_after  = hist.loc[future_dates[DAYS_AFTER], "Close"]

        # Calculate % change
        pct_change = (price_after - price_before) / price_before

        # Assign label based on threshold
        if pct_change > PRICE_THRESHOLD:
            label = "UP"
        elif pct_change < -PRICE_THRESHOLD:
            label = "DOWN"
        else:
            label = "FLAT"

        return {
            "price_before": round(float(price_before), 2),
            "price_after":  round(float(price_after), 2),
            "pct_change":   round(float(pct_change) * 100, 2),  # as percentage
            "label":        label
        }

    except Exception as e:
        return None


print("✅ Price label function defined")

# ── Quick test ─────────────────────────────────────────────────────────────────
print("\n🔍 Testing price label for Apple on 2023-11-02 (their Q4 2023 earnings)...")
test_label = get_price_label("AAPL", "2023-11-02")
if test_label:
    print(f"   Price before:  ${test_label['price_before']}")
    print(f"   Price after:   ${test_label['price_after']}")
    print(f"   % Change:      {test_label['pct_change']}%")
    print(f"   Label:         {test_label['label']}")
else:
    print("   Could not fetch price data")

✅ Price label function defined

🔍 Testing price label for Apple on 2023-11-02 (their Q4 2023 earnings)...
   Price before:  $175.51
   Price after:   $177.15
   % Change:      0.93%
   Label:         FLAT


## Cell 8 — Main Data Collection Loop

Now we put it all together.

For each company:
1. Get its CIK from SEC
2. Fetch all its 8-K filings in our date range
3. Download each filing text
4. Check if it's an earnings call
5. Extract prepared remarks
6. Get price label
7. Save to our dataset

**Note:** This will take 20-40 minutes to run for all 20 companies.
The `tqdm` progress bars will show you exactly how far along you are.
We save progress as we go so if it crashes, you don't lose work.

In [ ]:
# ── Main Data Collection Loop ─────────────────────────────────────────────────
# This is the main loop — it collects everything and saves to CSV

# Storage for all collected records
all_records = []

# Log file to track errors
error_log = []

print("🚀 Starting data collection...")
print(f"   Processing {len(COMPANIES)} companies from {START_YEAR} to {END_YEAR}")
print("   This will take approximately 20-40 minutes.")
print("-" * 60)

# ── Loop over each company ────────────────────────────────────────────────────
for company in tqdm(COMPANIES, desc="Companies"):
    ticker = company["ticker"]
    name   = company["name"]
    sector = company["sector"]

    print(f"\n📊 Processing {name} ({ticker})...")

    # Step 1: Get company CIK
    cik = get_company_cik(ticker)
    if not cik:
        error_log.append({"ticker": ticker, "error": "CIK not found"})
        continue

    # Step 2: Get all 8-K filings
    filings = get_8k_filings(cik, ticker, START_YEAR, END_YEAR)
    print(f"   Found {len(filings)} total 8-K filings")

    if not filings:
        continue

    earnings_count = 0

    # Step 3: Process each filing
    for filing in tqdm(filings, desc=f"  {ticker} filings", leave=False):

        # Download the filing text
        text = download_filing_text(
            filing["cik"],
            filing["accession"],
            filing["document"]
        )

        if not text:
            continue

        # Check if this is actually an earnings call
        if not is_earnings_call(text):
            continue  # skip — it's some other kind of 8-K announcement

        # Extract prepared remarks (most signal-rich part)
        transcript = extract_ceo_remarks(text)

        # Skip if transcript is too short to be useful
        if len(transcript.split()) < 200:
            continue

        # Get stock price label
        price_data = get_price_label(ticker, filing["date"])

        if not price_data:
            continue

        # Save the complete record
        record = {
            # Company info
            "ticker":        ticker,
            "company_name":  name,
            "sector":        sector,

            # Filing info
            "filing_date":   filing["date"],
            "accession":     filing["accession"],

            # The actual transcript text
            "transcript":    transcript,
            "word_count":    len(transcript.split()),

            # Price data and label
            "price_before":  price_data["price_before"],
            "price_after":   price_data["price_after"],
            "pct_change":    price_data["pct_change"],
            "label":         price_data["label"],   # ← THIS IS OUR TRAINING TARGET
        }

        all_records.append(record)
        earnings_count += 1

        # Save a text file of the transcript for inspection
        txt_path = RAW_DIR / f"{ticker}_{filing['date']}.txt"
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(f"Ticker: {ticker}\n")
            f.write(f"Date: {filing['date']}\n")
            f.write(f"Label: {price_data['label']} ({price_data['pct_change']}%)\n")
            f.write("=" * 60 + "\n")
            f.write(transcript)

        # Save progress every 10 records (in case of crash)
        if len(all_records) % 10 == 0:
            temp_df = pd.DataFrame(all_records)
            temp_df.to_csv(PROCESSED_DIR / "dataset_temp.csv", index=False)

    print(f"   ✅ Collected {earnings_count} earnings call transcripts")

print("\n" + "=" * 60)
print(f"🎉 Collection complete! Total records: {len(all_records)}")

🚀 Starting data collection...
   Processing 20 companies from 2018 to 2024
   This will take approximately 20-40 minutes.
------------------------------------------------------------


Companies:   0%|          | 0/20 [00:00<?, ?it/s]


📊 Processing Apple Inc (AAPL)...
   ❌ Error Connecting for AAPL: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb09e150>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing Microsoft Corp (MSFT)...
   ❌ Error Connecting for MSFT: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=MSFT&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb41afc0>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing Alphabet Inc (GOOGL)...
   ❌ Error Connecting for GOOGL: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /c

Companies:  70%|███████   | 14/20 [00:00<00:00, 131.46it/s]

   ❌ Error Connecting for MCD: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=MCD&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb41a330>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing Exxon Mobil (XOM)...
   ❌ Error Connecting for XOM: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=XOM&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb418b90>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing Chevron Corp (CVX)...
   ❌ Error Connecting for CVX: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=CVX&

Companies: 100%|██████████| 20/20 [00:00<00:00, 133.24it/s]

   ❌ Error Connecting for BA: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=BA&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb418fe0>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing Caterpillar Inc (CAT)...
   ❌ Error Connecting for CAT: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=CAT&type=&dateb=&owner=only&start=0&count=1000&output=xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7806cb41b020>: Failed to resolve 'api.sec.gov' ([Errno -2] Name or service not known)"))

📊 Processing AT&T Inc (T)...
   ❌ Error Connecting for T: HTTPSConnectionPool(host='api.sec.gov', port=443): Max retries exceeded with url: /cgi-bin/browse-edgar?action=getcompany&CIK=T&type=&da

In [ ]:
# Run this cell to check network connectivity to the SEC API server
!ping api.sec.gov

/bin/bash: line 1: ping: command not found


In [ ]:
# Run this cell to check network connectivity using curl
!curl -I api.sec.gov

curl: (6) Could not resolve host: api.sec.gov
